In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import joblib

from model import DemandModel

In [2]:
df = pd.read_csv("dataset/train.csv", index_col='Index')
df.head()

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
Index,,,,,,,,,,
0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [3]:
tv_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [4]:
print(len(tv_df), len(test_df))

61839 15460


### -----HYPERPARAMETER TUNING----- ###

In [5]:
params_base = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1
}

In [6]:
from optimizer import tune

study = tune(tv_df, params_base)
best_params = study.best_params
print(best_params)

c:\Users\prasa\Desktop\NITD CSE\GridLock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-06 02:37:46,170] A new study created in memory with name: no-name-1948ca62-cac4-4a94-b810-7eb9d1b309a9
Optuna Tuning: 100%|██████████| 50/50 [07:22<00:00,  8.85s/it, best=0.915]

Best R2: 0.9151396110725868
Best params: {'learning_rate': 0.005239119718113649, 'num_leaves': 234, 'max_depth': 10, 'min_data_in_leaf': 133, 'feature_fraction': 0.9630023044544022, 'bagging_fraction': 0.9503692489666424, 'bagging_freq': 1, 'lambda_l1': 0.17931093106925933, 'lambda_l2': 12.251484490316788, 'min_gain_to_split': 9.544089571948146e-05}
{'learning_rate': 0.005239119718113649, 'num_leaves': 234, 'max_depth': 10, 'min_data_in_leaf': 133, 'feature_fraction': 0.9630023044544022, 'bagging_fraction': 0.9503692489666424, 'bagging_freq': 1, 'lambda_l1': 0.17931093106925933, 'lambda_l2': 12.251484490316788, 'min_gain_to_split': 9.544089571948146e-05}


In [ ]:
from sklearn.model_selection import KFold

x = tv_df.copy()
y = tv_df["demand"]

kf = KFold(n_splits=4,
           shuffle=True,
           random_state=42)
oof = np.zeros(len(tv_df))

for fold, (tr, va) in enumerate(kf.split(x, y)):

    train_fold = tv_df.iloc[tr]
    val_fold = tv_df.iloc[va]

    model = DemandModel(best_params, verbose=True)
    model.fit(train_fold, val_fold)

    preds = model.predict(val_fold)
    oof[va] = preds

    print(f"Fold {fold + 1} R2:", r2_score(val_fold["demand"], preds))

print("OOF R2:", r2_score(y, oof))

Fold 1 R2: 0.9184138674564459
Fold 2 R2: 0.9136291730289576
Fold 3 R2: 0.9111243873737092
Fold 4 R2: 0.9108540432985133
OOF R2: 0.9136742811329309


### -----TESTING ROUTINE----- ###

In [8]:
test_y = test_df["demand"]
test_x = test_df.drop(columns=["demand"])
preds = model.predict(test_x)

print("Test R2:", r2_score(test_y, preds))

Test R2: 0.9147869147574683


In [9]:
joblib.dump(model.model, "model.pkl")
joblib.dump(model.pipeline, "pipeline.pkl")

['pipeline.pkl']

In [10]:
submission_test_df = pd.read_csv("dataset/test.csv")
predictions = model.predict(submission_test_df)

submission = pd.DataFrame({
    "Index": submission_test_df.index,
    "demand": predictions
})

submission.to_csv("dataset/submission.csv", index=False)